# Lección 10.5 — Transfer learning y fine-tuning: no empezar de cero

> **Módulo 10 · Visión por computadora (CNN)** · Nivel 4 — Avanzado
> ⏱️ Tiempo estimado: 100–120 min (lección con ejercicios intercalados ≈ 80 min · ejercicios finales ≈ 30 min)
> 📋 Requisitos previos: Lección 10.4 (ResNet), Lección 9.8 (data augmentation), Lección 4.4 (sobreajuste)

### 🎯 Objetivos
Al terminar esta lección podrás:
- Explicar qué es el **transfer learning** y por qué funciona (la jerarquía de características se reutiliza).
- Usar una **ResNet18 preentrenada** en ImageNet: para predecir, como **extractor de características** y con **fine-tuning**.
- Elegir la estrategia (congelar o ajustar, cuántas capas) según la **cantidad de datos** y su **parecido** con ImageNet.
- Evitar los errores clásicos: olvidar la normalización de ImageNet, `model.eval()`, tasa de aprendizaje demasiado alta.

### 🧭 Cómo usar este notebook
| Icono | Significa |
|---|---|
| 📖 | Lee: explicación del concepto |
| ▶️ | Ejecuta la celda y observa el resultado |
| 🔮 | **Predice** antes de ejecutar: escribe tu apuesta y luego compruébala |
| ✋ | **Tu turno**: ejercicio. Las respuestas están ocultas en "👀 Ver respuesta" |
| ✏️ | Modifica un valor en el código y vuelve a ejecutar |

> 💡 No necesitas saber programar. El código está para que **veas** los conceptos en acción; solo tendrás que cambiar números.
>
> 🖥️ Todo corre en CPU en pocos minutos (también en Google Colab sin GPU). Si tienes GPU, se usará sola.

## ▶️ Preparación
La primera vez se descargan **CIFAR-10** (≈170 MB: 60,000 fotos de 32 × 32 de 10 clases) y los pesos de **ResNet18** (≈45 MB). En Colab tarda uno o dos minutos.

In [ ]:
import time, copy
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torchvision import datasets, models, transforms as T
from sklearn.datasets import load_sample_image

plt.rcParams["figure.figsize"] = (8, 4.5)
torch.manual_seed(0)

cifar = datasets.CIFAR10(root="data", train=True, download=True)
cifar_test = datasets.CIFAR10(root="data", train=False, download=True)
pesos = models.ResNet18_Weights.DEFAULT                   # pesos entrenados en ImageNet
print("✅ Todo listo")

---
## 1. 📖 La intuición: no aprender a cocinar desde cero

Una chef experta en cocina italiana quiere aprender cocina tailandesa. **No** empieza aprendiendo a sostener un cuchillo o a encender la estufa: eso ya lo sabe. Solo aprende lo **nuevo**: los ingredientes, las especias, las combinaciones. En unas semanas cocina bien platos tailandeses; alguien que empieza desde cero tardaría años.

El **transfer learning** (aprendizaje por transferencia) hace lo mismo con las redes:
1. Toma una red que ya aprendió a "ver" con **millones** de fotos (ResNet entrenada en ImageNet: 1.2 millones de fotos, 1,000 clases).
2. Reutiliza lo que ya sabe: sus filtros detectan bordes, texturas, formas, partes de objetos… (la **jerarquía** de la Lección 10.3).
3. Solo le enseña lo **nuevo**: tus clases, con tus (pocas) fotos.

```
              ImageNet (1.2 M fotos)                         Tus datos (400 fotos)
 ┌──────────────────────────────────────┬──────────┐      ┌──────────────────────────────────────┬──────────┐
 │ conv1 → layer1 → layer2 → layer3 → layer4 │ fc 1000 │  ─►  │ conv1 → layer1 → layer2 → layer3 → layer4 │ fc   4   │
 │   bordes  texturas  formas   partes  objetos│         │      │      (se reutiliza: congelado ❄️)         │ (nueva🔥)│
 └──────────────────────────────────────┴──────────┘      └──────────────────────────────────────┴──────────┘
```

### ✋ Ejercicio 1 — ¿Qué se puede reutilizar?
Una red entrenada para reconocer **animales** se va a reutilizar para clasificar **radiografías de tórax**. ¿Qué capas crees que siguen siendo útiles: las primeras (bordes, texturas) o las últimas (orejas, hocicos, ruedas)? ¿Por qué?

✍️ *Mi respuesta:*

<details><summary><b>👀 Ver respuesta</b></summary>

Las **primeras** capas son útiles casi para cualquier imagen: bordes, contrastes y texturas existen tanto en fotos de perros como en radiografías. Las **últimas** son muy específicas de la tarea original ("hocico de perro") y sirven menos. Por eso, cuanto más **distinta** es tu tarea de ImageNet, más capas conviene **re-entrenar** (Sección 5).
</details>

---
## 2. 📖 Una red que ya sabe ver

Primero, usemos ResNet18 **tal cual**, sin entrenar nada. `torchvision` trae, junto con los pesos, la receta de preparación de las imágenes (`pesos.transforms()`) y los nombres de las 1,000 clases.

In [ ]:
resnet = models.resnet18(weights=pesos).eval()         # eval(): modo "examen" (Sección 6)
preparar = pesos.transforms()
categorias = pesos.meta["categories"]
print(preparar)

def top5(img_np):
    x = preparar(torch.tensor(img_np).permute(2, 0, 1)).unsqueeze(0)   # (1, 3, 224, 224)
    with torch.no_grad():
        probs = resnet(x).softmax(1)[0]
    return [(categorias[i], p.item()) for p, i in zip(*probs.topk(5))]

### 🔮 Ejercicio 2 — ¿Qué dirá la red?
Le mostraremos dos fotos incluidas en scikit-learn: el **templo** chino de la Lección 10.1 y una **flor** (una dalia). ImageNet tiene clases como *palace*, *monastery*, *daisy*… pero **no** tiene "templo chino" ni "dalia".

¿Qué crees que responderá para cada una? ✍️ *Mis apuestas:*

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
for a, nombre in zip(ax, ["china.jpg", "flower.jpg"]):
    img = load_sample_image(nombre)
    a.imshow(img); a.axis("off")
    a.set_title("\n".join(f"{c}: {p:.0%}" for c, p in top5(img)[:3]), fontsize=9)
plt.show()

<details><summary><b>👀 Ver respuesta</b></summary>

- **Templo:** *bell cote* (espadaña, ≈53 %), *palace* (≈31 %), *monastery*… Muy razonable: elige las clases de ImageNet más parecidas.
- **Flor:** ¡*honeycomb* (panal de abejas, ≈67 %)! Y luego hongos (*hen-of-the-woods*, *stinkhorn*…). La dalia tiene un patrón **repetitivo de celdas**, como un panal. Las CNN se fijan mucho en la **textura**, y como "dalia" no es una de sus 1,000 clases, elige la textura más parecida.

Moraleja: la red **ya sabe ver** (bordes, texturas, formas), pero **solo conoce sus 1,000 clases**. Para que reconozca **tus** clases hay que adaptarla: eso es el *transfer learning*.
</details>

---
## 3. 📖 Nuestro problema: pocos datos

Imagina que una protectora de animales quiere clasificar automáticamente fotos de **gatos, ciervos, perros y caballos**, pero solo tiene **100 fotos etiquetadas de cada uno** (400 en total). Usaremos 4 clases de CIFAR-10 para simularlo, y 200 fotos por clase para la prueba.

In [ ]:
CLASES = [3, 4, 5, 7]                                  # índices en CIFAR-10
NOMBRES = ["gato", "ciervo", "perro", "caballo"]
N_POR_CLASE = 100                                      # ✏️ (lo usarás en los ejercicios finales)

def subconjunto(ds, n):
    y = np.array(ds.targets)
    idx = np.concatenate([np.where(y == c)[0][:n] for c in CLASES])
    X = torch.tensor(ds.data[idx]).permute(0, 3, 1, 2).float() / 255    # (N, 3, 32, 32), 0–1
    Y = torch.tensor([CLASES.index(c) for c in y[idx]])
    return X, Y

X_tr, y_tr = subconjunto(cifar, N_POR_CLASE)
X_te, y_te = subconjunto(cifar_test, 200)
print("Entrenamiento:", tuple(X_tr.shape), "· Prueba:", tuple(X_te.shape))

fig, ax = plt.subplots(2, 8, figsize=(13, 3.6))
for k, a in enumerate(ax.flat):
    i = (k % 4) * N_POR_CLASE + k // 4
    a.imshow(X_tr[i].permute(1, 2, 0)); a.set_title(NOMBRES[y_tr[i]], fontsize=8); a.axis("off")
plt.show()

Usaremos estas dos funciones en toda la lección (son las mismas ideas de las lecciones anteriores):

In [ ]:
def entrenar(modelo, X, y, epocas, lr, parametros=None, aumentar=False, lote=32):
    """Entrena con Adam; si aumentar=True, voltea al azar la mitad de las fotos (data augmentation)."""
    torch.manual_seed(0)
    opt = torch.optim.Adam(parametros or modelo.parameters(), lr=lr)
    for _ in range(epocas):
        modelo.train(); orden = torch.randperm(len(X))
        for i in range(0, len(X), lote):
            idx = orden[i:i + lote]; xb = X[idx]
            if aumentar:
                xb = torch.where(torch.rand(len(xb), 1, 1, 1) < 0.5, xb.flip(3), xb)
            perdida = nn.functional.cross_entropy(modelo(xb), y[idx])
            opt.zero_grad(); perdida.backward(); opt.step()
    return modelo

def exactitud(modelo, X, y):
    modelo.eval()
    with torch.no_grad():
        pred = torch.cat([modelo(X[i:i + 200]).argmax(1) for i in range(0, len(X), 200)])
    return (pred == y).float().mean().item()

### 🔮 Ejercicio 3 — La línea base: una CNN desde cero
Primero, lo que harías sin *transfer learning*: entrenar una CNN pequeña (3 bloques, como en la Lección 10.3) **desde cero** con las 400 fotos, 30 épocas. Hay 4 clases, así que adivinar al azar da 25 %.

¿Qué exactitud en prueba esperas? ¿Habrá sobreajuste? ✍️ *Mi apuesta:*

▶️ Tarda entre 20 y 40 s.

In [ ]:
torch.manual_seed(0)
cnn_cero = nn.Sequential(
    nn.Conv2d(3, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
    nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
    nn.Conv2d(64, 64, 3, padding=1), nn.ReLU(),
    nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.Linear(64, 4))
inicio = time.time()
entrenar(cnn_cero, X_tr, y_tr, epocas=30, lr=1e-3)
print(f"CNN desde cero → entrenamiento {exactitud(cnn_cero, X_tr, y_tr):.1%} · prueba {exactitud(cnn_cero, X_te, y_te):.1%} · {time.time() - inicio:.0f} s")

<details><summary><b>👀 Ver respuesta</b></summary>

Apenas **≈44 %** en prueba (≈51 % en entrenamiento). Mejor que el azar (25 %), pero muy pobre. Con 400 fotos de 32 × 32 la red no tiene suficientes ejemplos para aprender por sí sola qué es una "oreja" o una "pata", y distinguir gatos de perros es difícil incluso para personas a esa resolución. Con más épocas o una red más grande solo conseguirías **sobreajustar**.
</details>

---
## 4. 📖 Cómo funciona: dos formas de transferir

### Estrategia A — Extractor de características (congelar todo ❄️)
1. Quita la última capa (`fc`, la que decide entre las 1,000 clases de ImageNet).
2. Pasa tus fotos por el resto de la red **sin entrenarla**: cada foto se convierte en un vector de **512 números** que describe lo que la red "ve" (un *embedding*).
3. Entrena **solo** una capa nueva y pequeña (`nn.Linear(512, 4)`) sobre esos vectores.

Es rapidísimo: la parte cara (la ResNet) se calcula **una sola vez** por foto.

> 📏 ResNet se entrenó con fotos de 224 × 224. Nuestras fotos son de 32 × 32, así que las **agrandamos** a 112 × 112 (un compromiso entre calidad y velocidad en CPU) y aplicamos la **normalización de ImageNet**: restar la media y dividir entre la desviación estándar de cada canal, exactamente como se hizo al entrenarla.

In [ ]:
normalizar_imagenet = T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])

def preparar_lote(X, lado=112, normalizar=True):
    X = nn.functional.interpolate(X, size=lado, mode="bilinear", align_corners=False)   # 32 → 112 px
    return normalizar_imagenet(X) if normalizar else X

extractor = models.resnet18(weights=pesos)
extractor.fc = nn.Identity()          # "quitamos" la última capa: ahora devuelve los 512 números
extractor.eval()

def caracteristicas(X, normalizar=True):
    with torch.no_grad():
        return torch.cat([extractor(preparar_lote(X[i:i + 100], normalizar=normalizar)) for i in range(0, len(X), 100)])

inicio = time.time()
F_tr, F_te = caracteristicas(X_tr), caracteristicas(X_te)
print(f"Cada foto ahora es un vector: {tuple(F_tr.shape)} · {time.time() - inicio:.0f} s")

### 🔮 Ejercicio 4 — ¿Cuánto ayuda?
Ahora entrenamos **solo** una capa lineal de 512 → 4 (¡2,052 parámetros!) sobre esos vectores. La CNN desde cero, con ≈57,000 parámetros entrenados, sacó ≈44 %. ¿Cuánto esperas con esta capa diminuta? ✍️ *Mi apuesta:*

In [ ]:
torch.manual_seed(0)
cabeza = nn.Linear(512, 4)
entrenar(cabeza, F_tr, y_tr, epocas=100, lr=1e-3)
print(f"Parámetros entrenados: {sum(p.numel() for p in cabeza.parameters()):,} (CNN desde cero: {sum(p.numel() for p in cnn_cero.parameters()):,})")
print(f"Extractor + cabeza lineal → entrenamiento {exactitud(cabeza, F_tr, y_tr):.1%} · prueba {exactitud(cabeza, F_te, y_te):.1%}")

<details><summary><b>👀 Ver respuesta</b></summary>

¡**≈78 %** en prueba! Casi el **doble** que la CNN desde cero, entrenando 28 veces menos parámetros y en un par de segundos (la parte lenta fue calcular las características una vez).

La razón: los 512 números ya describen la foto con conceptos útiles ("hay pelaje", "hay patas largas", "hay hocico alargado"…) aprendidos con 1.2 millones de fotos. Separar 4 clases con esa descripción es fácil para una simple capa lineal.

Fíjate en que la exactitud de **entrenamiento** es del 100 %: con tan pocas fotos, hasta la cabeza lineal memoriza. Aun así generaliza mucho mejor que la red desde cero.
</details>

### Estrategia B — Fine-tuning (ajuste fino 🔥)
En lugar de congelar toda la red, **descongelamos** también las últimas capas (por ejemplo `layer4`) y las entrenamos un poco, con una **tasa de aprendizaje pequeña**, para que se adapten a nuestras clases sin "olvidar" lo que sabían.

Receta habitual:
1. Primero entrena solo la cabeza (lo que ya hicimos).
2. Luego descongela `layer4` y sigue entrenando con un `lr` **10 veces menor** (aquí 1e-4) y *data augmentation*.

### 🔮 Ejercicio 5 — ¿Fine-tuning gana?
Con **solo 100 fotos por clase**, ¿crees que ajustar `layer4` (8.4 millones de parámetros, Lección 10.4) mejorará el ≈78 % del extractor? ✍️ *Mi apuesta:*

▶️ Tarda entre 30 s y 1.5 min (ahora sí hay que pasar las fotos por toda la red en cada época).

In [ ]:
fino = models.resnet18(weights=pesos)
fino.fc = copy.deepcopy(cabeza)                      # partimos de la cabeza ya entrenada
for p in fino.parameters():
    p.requires_grad = False                          # ❄️ congelamos todo…
for p in list(fino.layer4.parameters()) + list(fino.fc.parameters()):
    p.requires_grad = True                           # 🔥 …menos layer4 y la cabeza

entrenables = [p for p in fino.parameters() if p.requires_grad]
print(f"Parámetros entrenables: {sum(p.numel() for p in entrenables):,} de {sum(p.numel() for p in fino.parameters()):,}")

P_tr, P_te = preparar_lote(X_tr), preparar_lote(X_te)
inicio = time.time()
entrenar(fino, P_tr, y_tr, epocas=3, lr=1e-4, parametros=entrenables, aumentar=True)
print(f"Fine-tuning de layer4 → prueba {exactitud(fino, P_te, y_te):.1%} · {time.time() - inicio:.0f} s")

<details><summary><b>👀 Ver respuesta</b></summary>

No: queda en **≈76 %**, prácticamente igual (o un poco peor) que el extractor (≈78 %), y tardó mucho más.

Con 400 fotos no hay suficiente información para ajustar millones de parámetros sin riesgo de sobreajustar: la red preentrenada ya era muy buena y "moverla" con tan pocos datos no aporta. El *fine-tuning* brilla cuando tienes **más datos** (miles de fotos) o cuando tus imágenes son **muy distintas** de ImageNet (radiografías, satélite…). Con pocos datos parecidos a ImageNet, **congelar** suele ser la mejor opción. Es justo la tabla de la Sección 5.
</details>

---
## 5. 📖 Hiperparámetros y estrategia: ¿cuánto congelo?

La decisión depende de dos preguntas: **¿cuántos datos tengo?** y **¿se parecen a ImageNet?**

| | Datos **parecidos** a ImageNet (fotos de objetos, animales…) | Datos **distintos** (radiografías, satélite, microscopio…) |
|---|---|---|
| **Pocos datos** (cientos) | ❄️ Extractor de características: congela todo, entrena solo la cabeza | ❄️ Extractor (quizás de capas intermedias); *augmentation* fuerte; considera conseguir más datos |
| **Muchos datos** (miles o más) | 🔥 *Fine-tuning* de las últimas etapas (`layer3`–`layer4`) | 🔥 *Fine-tuning* de toda la red (o casi) |

| Hiperparámetro | Qué controla | Si lo **subes** ⬆️ | Si lo **bajas** ⬇️ | Punto de partida |
|---|---|---|---|---|
| **Capas descongeladas** | Cuánto se adapta la red | Más adaptación; más riesgo de sobreajuste y más lento | Más estable y rápido; menos adaptación | Solo la cabeza; luego `layer4` si hay datos |
| **`lr` del fine-tuning** | Cuánto se mueven los pesos preentrenados | "Olvida" lo aprendido (*olvido catastrófico*) | Aprende muy despacio | **1e-4** (10–100 veces menor que desde cero) |
| **`lr` de la cabeza** | Velocidad de la capa nueva | Inestable | Lento | 1e-3 |
| **Épocas** | Cuánto se ajusta | Sobreajuste (vigila la validación) | Subajuste | Pocas: 3–10 |
| **Resolución de entrada** | Detalle y costo | Más exactitud (hasta ~224); mucho más lento | Más rápido; pierde detalle | 224 (aquí 112 por la CPU) |
| ***Data augmentation*** | Variedad de los datos | Más robusto; si es excesiva, cambia la etiqueta | Más sobreajuste | Volteo horizontal + recorte aleatorio |

### 🔮 Ejercicio 6 — ¿Cuántas fotos necesito?
Como ya calculamos las características de las 400 fotos, entrenar la cabeza es casi instantáneo. Veamos cómo cambia la exactitud con **5, 10, 25, 50 y 100 fotos por clase**. Antes de ejecutar: con solo **5 fotos** de cada animal (20 en total), ¿qué exactitud esperas?

In [ ]:
for n in [5, 10, 25, 50, 100]:                     # ✏️
    idx = torch.cat([torch.arange(k * N_POR_CLASE, k * N_POR_CLASE + n) for k in range(4)])
    torch.manual_seed(0)
    cab_n = entrenar(nn.Linear(512, 4), F_tr[idx], y_tr[idx], epocas=100, lr=1e-3)
    acc = exactitud(cab_n, F_te, y_te)
    print(f"{n:>3} fotos por clase → prueba {acc:.1%}  " + "█" * int(acc * 40))

<details><summary><b>👀 Ver respuesta</b></summary>

Con solo **5 fotos por clase** ya se supera claramente al azar y se acerca (o supera) a la CNN desde cero entrenada con **100** por clase (≈44 %). La exactitud sube rápido con las primeras fotos y luego cada foto extra aporta menos (rendimientos decrecientes). Los valores exactos dependen de qué fotos tocan, pero la tendencia es clara: **con transfer learning, unas pocas decenas de ejemplos por clase ya dan un modelo útil**. Es la razón por la que es la opción por defecto en la industria.
</details>

### 📖 Data augmentation para imágenes
En el *fine-tuning* usamos volteos horizontales. `torchvision.transforms` trae muchas más transformaciones (Lección 9.8). Mira qué hacen sobre una misma foto:

In [ ]:
aumento = T.Compose([
    T.RandomResizedCrop(32, scale=(0.6, 1.0)),       # recorte aleatorio y reescalado
    T.RandomHorizontalFlip(),                        # espejo
    T.ColorJitter(brightness=0.4, contrast=0.4, saturation=0.4),   # cambios de luz y color
])                                                   # ✏️ añade T.RandomRotation(30) o T.RandomVerticalFlip()
torch.manual_seed(1)
foto = X_tr[0]
fig, ax = plt.subplots(1, 8, figsize=(14, 2))
ax[0].imshow(foto.permute(1, 2, 0)); ax[0].set_title("original", fontsize=8)
for a in ax[1:]:
    a.imshow(aumento(foto).permute(1, 2, 0).clamp(0, 1))
for a in ax: a.axis("off")
plt.show()

### ✋ Ejercicio 7 — ¿Qué aumentos tienen sentido?
Para cada problema, ¿usarías **volteo horizontal**, **volteo vertical** y **rotaciones de 90°**?
1. Gatos, ciervos, perros y caballos.
2. Fotos de satélite de campos de cultivo.
3. Dígitos escritos a mano.

✍️ *Mis respuestas:*

<details><summary><b>👀 Ver respuesta</b></summary>

1. **Volteo horizontal sí** (un perro mirando a la izquierda sigue siendo un perro); volteo vertical y rotaciones de 90° **no** (no hay caballos de cabeza en la vida real, y el modelo gastaría capacidad en casos que nunca verá).
2. **Los tres sí**: desde el satélite no hay "arriba" ni "abajo"; un campo girado sigue siendo un campo.
3. **Ninguno** (o rotaciones muy pequeñas, de ±10°): un 6 girado se convierte en un 9, y un 2 en espejo deja de ser un 2. La regla: **un aumento nunca debe cambiar la etiqueta** (lo viste en la Lección 10.1 al voltear dígitos).
</details>

---
## 6. 📖 Errores comunes y señales de alerta

| 🚨 Síntoma | Causa probable | Qué probar |
|---|---|---|
| El modelo preentrenado da resultados mucho peores de lo esperado | Olvidaste la **normalización de ImageNet** o usas otra resolución/orden de canales | Usa `pesos.transforms()` o `T.Normalize` con la media y la desviación de ImageNet |
| Predicciones distintas cada vez que evalúas la misma foto | No pusiste `model.eval()`: BatchNorm y Dropout siguen en modo entrenamiento | `model.eval()` antes de evaluar; `model.train()` para entrenar |
| Tras unas pocas iteraciones de fine-tuning la exactitud se desploma | `lr` demasiado alto: destruye lo aprendido (olvido catastrófico) | `lr` 10–100 veces menor; entrena primero la cabeza |
| 100 % en entrenamiento, mucho menos en prueba | Muy pocos datos para lo que estás ajustando | Congela más capas, más *augmentation*, más datos |
| *Error: size mismatch for fc* o 1,000 salidas en vez de tus clases | No cambiaste la última capa | `modelo.fc = nn.Linear(512, n_clases)` |
| El fine-tuning es lentísimo | Calculas en cada época la parte congelada | Si todo está congelado, **precalcula** las características una vez (Sección 4) |

### 🔮 Ejercicio 8 — Sin normalizar
Repetimos el extractor de características, pero **sin** la normalización de ImageNet (las fotos van de 0 a 1 en lugar de estar centradas como en el entrenamiento original). ¿Cuánto crees que se pierde? ▶️ Entre 15 y 45 s.

In [ ]:
F_tr_sin, F_te_sin = caracteristicas(X_tr, normalizar=False), caracteristicas(X_te, normalizar=False)
torch.manual_seed(0)
cabeza_sin = entrenar(nn.Linear(512, 4), F_tr_sin, y_tr, epocas=100, lr=1e-3)
print(f"Con normalización de ImageNet: {exactitud(cabeza, F_te, y_te):.1%}")
print(f"Sin normalización            : {exactitud(cabeza_sin, F_te_sin, y_te):.1%}")

<details><summary><b>👀 Ver respuesta</b></summary>

Se pierden **≈11 puntos** (≈78 % → ≈67 %). La red aprendió sus filtros con fotos normalizadas de una manera concreta; si le das números en otra escala, los mapas de características se distorsionan (es como si a la chef le cambiaran las unidades de la receta sin avisar). No da ningún error: **solo funciona peor en silencio**, que es lo más peligroso. Usa siempre la misma preparación con la que se entrenó el modelo (`pesos.transforms()`).
</details>

### 🔮 Ejercicio 9 — El modo examen
`model.train()` y `model.eval()` cambian el comportamiento de BatchNorm (y Dropout). ¿Qué pasará si calculamos las características de las **mismas** 8 fotos en modo `train()`, en lotes distintos? ¿Y si después volvemos a `eval()`: saldrá lo mismo que al principio?

In [ ]:
copia = copy.deepcopy(extractor)          # trabajamos con una copia para no estropear el extractor
ocho = preparar_lote(X_te[:8])
with torch.no_grad():
    copia.eval();  a = copia(ocho)
    copia.train(); b = copia(ocho); c = copia(ocho[:4])        # mismo modo train, otro lote
    copia.eval();  d = copia(ocho)                              # de vuelta al modo examen
print(f"eval vs. train (mismas fotos)        : diferencia media {(a - b).abs().mean():.3f}")
print(f"train con lote de 8 vs. lote de 4    : diferencia media {(b[:4] - c).abs().mean():.3f}")
print(f"eval antes vs. eval después de train : diferencia media {(a - d).abs().mean():.3f}   ← ¿no debería ser 0?")

<details><summary><b>👀 Ver respuesta</b></summary>

En modo `train()`, BatchNorm normaliza con las estadísticas **del lote actual** en lugar de las aprendidas: las características cambian respecto a `eval()`, e incluso cambian según **qué otras fotos** van en el mismo lote. ¡La predicción de una foto dependería de sus compañeras de lote! Y la sorpresa: al volver a `eval()`, ¡el resultado **ya no es el del principio**! En modo `train()`, cada pasada hace que BatchNorm **actualice** sus estadísticas internas (sus "promedios de referencia") con esas 8 fotos: con solo pasar dos lotes en modo entrenamiento **modificamos la red sin querer**, aunque no hubo ningún optimizador. Por eso trabajamos con una copia. Regla: `eval()` para predecir o extraer características, `train()` solo al entrenar.
</details>

---
## 7. 📋 Ficha del método: transfer learning

| | Transfer learning con una CNN preentrenada |
|---|---|
| **Intuición** | No aprender a ver desde cero: reutilizar una red que ya vio millones de fotos y enseñarle solo lo nuevo |
| **Dos sabores** | ❄️ **Extractor de características** (congelar y entrenar una cabeza nueva) · 🔥 **Fine-tuning** (descongelar las últimas capas, `lr` pequeño) |
| **Hiperparámetros clave** | Capas descongeladas, `lr` (pequeño para lo preentrenado), épocas, resolución, *augmentation* |
| **✅ Fortalezas** | Funciona con **muy pocos datos** · Entrena rápido · Mejor exactitud que desde cero · Es el estándar de la industria |
| **❌ Debilidades** | Hereda los sesgos del dataset original · Si tus imágenes son muy distintas, transfiere menos · El modelo base puede ser grande para un celular |
| **Úsalo cuando…** | Casi siempre que trabajes con imágenes y no tengas millones de ejemplos |
| **Evítalo cuando…** | Tus datos no se parecen a nada (p. ej. espectrogramas de radar) **y** tienes muchísimos datos; aun así, suele ayudar como punto de partida |

## 8. 🧠 Resumen en 5 puntos
1. Las primeras capas de una CNN aprenden cosas **generales** (bordes, texturas) que sirven para casi cualquier imagen: se pueden **reutilizar**.
2. **Extractor de características**: congela la red, quita la última capa y entrena solo una cabeza nueva. Con 100 fotos por clase pasamos de ≈44 % (desde cero) a ≈78 %.
3. **Fine-tuning**: descongela las últimas capas con un `lr` pequeño. Vale la pena con **más datos** o con datos **distintos** de ImageNet.
4. Prepara las fotos **exactamente** como en el entrenamiento original (tamaño, normalización) y usa `model.eval()` al predecir.
5. El *data augmentation* debe **conservar la etiqueta**: el volteo horizontal sirve para animales, no para dígitos.

---
## 9. 🏋️ Ejercicios finales

### 9.1 Repaso rápido (opción múltiple)

**1.** En la estrategia de "extractor de características", ¿qué se entrena?
- a) Toda la red · b) Solo `layer4` · c) Solo la cabeza nueva · d) Nada

**2.** ¿Por qué en el fine-tuning se usa un `lr` más pequeño que al entrenar desde cero?
- a) Para que el entrenamiento sea más rápido
- b) Para no destruir lo que la red ya aprendió
- c) Porque Adam no acepta valores grandes
- d) Porque las imágenes son más pequeñas

**3.** Tienes 300 radiografías y una ResNet preentrenada en ImageNet. La opción más sensata para empezar es…
- a) Entrenar una ResNet50 desde cero
- b) Extractor de características (y quizás ajustar poco a poco), con *augmentation*
- c) Fine-tuning de toda la red con `lr = 0.1`
- d) Usar la ResNet sin cambios, con sus 1,000 clases

**4.** Tu modelo preentrenado da 20 puntos menos de lo esperado y no hay ningún error. ¿Qué revisas primero?
- a) La normalización y el tamaño de las imágenes · b) El número de épocas · c) El optimizador · d) La semilla aleatoria

**5.** ¿Qué *augmentation* NO tiene sentido para clasificar señales de tránsito ("girar a la izquierda" / "girar a la derecha")?
- a) Cambios de brillo · b) Recortes pequeños · c) Volteo horizontal · d) Ligero desenfoque

✍️ *Mis respuestas:* 1 → _ · 2 → _ · 3 → _ · 4 → _ · 5 → _

<details><summary><b>👀 Ver respuestas</b></summary>

1. **c)** El resto está congelado.
2. **b)** Pasos grandes "borran" los pesos preentrenados (olvido catastrófico).
3. **b)** Pocos datos → congelar; como son distintos de ImageNet, puede valer la pena ajustar algunas capas con cuidado y medir con validación.
4. **a)** Es la causa más común de un modelo preentrenado que "funciona peor en silencio".
5. **c)** ¡Voltear una flecha a la izquierda la convierte en una flecha a la derecha! El aumento cambiaría la etiqueta.
</details>

### 9.2 Preguntas para pensar (sin ejecutar)

**A.** La ResNet preentrenada dijo que la dalia era un "panal". ¿Significa eso que no sirve como punto de partida para un clasificador de **flores**? Argumenta.

**B. Caso 💼** Una empresa de seguros quiere clasificar fotos de daños en autos en **leve / moderado / grave**. Tiene 2,000 fotos etiquetadas. El equipo A propone una CNN desde cero "para que aprenda exactamente nuestro problema". El equipo B propone *transfer learning* con ResNet18. ¿A quién apoyas? Describe en 4 pasos cómo lo harías y qué medirías.

✍️ *Mis respuestas:*

<details><summary><b>👀 Ver respuestas</b></summary>

**A.** Al contrario: sirve **muchísimo**. Que se equivoque de etiqueta solo significa que "dalia" no estaba entre sus 1,000 clases. Lo que nos importa son sus **características internas**: detectó la textura de pétalos repetidos, la forma circular, los colores… Con una cabeza nueva entrenada con fotos de flores etiquetadas, esas características permiten separar dalias, rosas y girasoles con pocos ejemplos. (De hecho, clasificar flores es un ejemplo clásico de *transfer learning*.)

**B.** Al **equipo B**. Con 2,000 fotos, una CNN desde cero sobreajustaría y daría peores resultados. Plan:
1. Separar entrenamiento / validación / prueba (por **siniestro**, no por foto, para evitar fugas si hay varias fotos del mismo auto, Lección 2.6).
2. Extractor de características con ResNet18 + cabeza de 3 clases → línea base rápida.
3. *Fine-tuning* de `layer4` (y quizá `layer3`) con `lr` pequeño y *augmentation* razonable (volteo horizontal, brillo; **no** recortes que eliminen el daño), comparando en validación.
4. Evaluar en prueba una sola vez, con matriz de confusión y *recall* por clase: confundir "grave" con "leve" probablemente cuesta mucho más que al revés (Lección 4.8), y revisar con CAM (Lección 10.6) que el modelo mira el daño y no el fondo.
</details>

### 9.3 🧩 Práctica integradora: "Diagnostica tus errores"

Con el mejor modelo de la lección (extractor + cabeza lineal), vas a hacer el análisis que hace cualquier equipo antes de entregar un modelo.

**Paso 1 🔮 — ¿Qué clases se confunden?** Antes de ejecutar: ¿qué pareja de animales crees que confunde más el modelo?

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

with torch.no_grad():
    probs_te = cabeza(F_te).softmax(1)
pred_te = probs_te.argmax(1)
print(confusion_matrix(y_te, pred_te))
ConfusionMatrixDisplay(confusion_matrix(y_te, pred_te), display_labels=NOMBRES).plot(cmap="Blues", colorbar=False)
plt.title("Filas = clase real · columnas = predicción"); plt.show()

<details><summary><b>👀 Ver respuesta</b></summary>

La confusión más grande es entre **gato y perro** (en las dos direcciones): a 32 × 32 píxeles, un gato y un perro pequeños se parecen mucho (pelaje, orejas, cuatro patas, fondos de interior). Ciervo y caballo también se confunden algo entre sí (animales grandes, patas largas, fondos de exterior). En cambio, rara vez confunde un gato con un caballo.
</details>

**Paso 2 ✋ — *Recall* por clase.** Calcula qué porcentaje de cada animal se reconoce correctamente. ¿Cuál es la clase más difícil?

In [ ]:
for k, nombre in enumerate(NOMBRES):
    mascara = y_te == k
    print(f"{nombre:<8} → recall {(pred_te[mascara] == k).float().mean():.0%}")

<details><summary><b>👀 Ver respuesta</b></summary>

Las clases más difíciles son **gato** y **perro**, con un *recall* claramente menor que ciervo y caballo. Si la protectora trabaja sobre todo con gatos y perros, la exactitud global (≈78 %) es **optimista** para su caso de uso: siempre mira las métricas **por clase** (Lección 4.2).
</details>

**Paso 3 ▶️ — Los errores más "seguros".** Los errores más preocupantes son aquellos en los que el modelo se equivoca **con mucha confianza**. Mostremos los 8 peores.

In [ ]:
errores = torch.where(pred_te != y_te)[0]
seguridad = probs_te[errores, pred_te[errores]]
peores = errores[seguridad.argsort(descending=True)[:8]]

fig, ax = plt.subplots(1, 8, figsize=(15, 2.6))
for a, i in zip(ax, peores):
    a.imshow(X_te[i].permute(1, 2, 0)); a.axis("off")
    a.set_title(f"real: {NOMBRES[y_te[i]]}\ndice: {NOMBRES[pred_te[i]]} ({probs_te[i, pred_te[i]]:.0%})", fontsize=8)
plt.show()
print(f"Errores totales: {len(errores)} de {len(y_te)}")

✍️ Mira las fotos: ¿son errores "tontos" o comprensibles? ¿Ves alguna foto mal etiquetada o casi imposible de clasificar a esa resolución?

<details><summary><b>👀 Ver respuesta</b></summary>

Normalmente verás errores **comprensibles**: animales de espaldas, muy oscuros, recortados, diminutos o con fondos que confunden (un perro sobre pasto de "ciervo"). Algunas fotos de CIFAR-10 son difíciles incluso para una persona a 32 × 32. Este análisis sirve para decidir el siguiente paso: ¿más fotos de cierto tipo?, ¿más resolución?, ¿revisar etiquetas? Mirar los errores **siempre** enseña algo que la métrica no dice.
</details>

**Paso 4 ✏️ — Mejora la resolución.** Las fotos se agrandaron a 112 × 112. ¿Qué pasa con 64 y con 160? Cambia `LADO` (▶️ con 160 tarda de 30 s a 1 min en CPU). ¿Vale la pena el costo?

In [ ]:
LADO = 64      # ✏️ prueba 64, 112 y 160

inicio = time.time()
with torch.no_grad():
    Ftr_l = torch.cat([extractor(preparar_lote(X_tr[i:i + 100], lado=LADO)) for i in range(0, len(X_tr), 100)])
    Fte_l = torch.cat([extractor(preparar_lote(X_te[i:i + 100], lado=LADO)) for i in range(0, len(X_te), 100)])
torch.manual_seed(0)
cab_l = entrenar(nn.Linear(512, 4), Ftr_l, y_tr, epocas=100, lr=1e-3)
print(f"Lado {LADO} px → prueba {exactitud(cab_l, Fte_l, y_te):.1%} · {time.time() - inicio:.0f} s")

<details><summary><b>👀 Ver respuesta</b></summary>

Con **64 px** la exactitud baja a ≈70 % (frente a ≈78 % con 112): la red se entrenó viendo objetos grandes, de ~224 px, y a 64 sus filtros "no encajan" tan bien. Con **160 px** sube a ≈82 %, pero el cálculo de características tarda unas 2 veces más que con 112. Es el compromiso de la Lección 10.4: el costo crece con los píxeles. En GPU usarías directamente 224.
</details>

**Paso 5 ✋ — Informe.** Escribe 3 frases para la protectora (que no sabe ML): qué tan bien funciona el modelo, dónde falla y qué recomiendas.

✍️ *Mi informe:*

<details><summary><b>👀 Ver respuesta</b></summary>

Algo como: *"Con solo 100 fotos de cada animal, el sistema acierta en unas 8 de cada 10 fotos nuevas, el doble de bien que una red entrenada desde cero. Donde más se equivoca es al distinguir gatos de perros, sobre todo en fotos oscuras, de espaldas o muy pequeñas. Recomendamos usarlo como sugerencia que una persona confirma, conseguir más fotos de gatos y perros y usar fotos de mayor calidad que las de prueba (32 × 32 píxeles)."*
</details>

---
## 📚 Para profundizar
- 📘 **Tutorial oficial:** PyTorch, *Transfer Learning for Computer Vision Tutorial* (hormigas vs. abejas con 240 fotos).
- 📘 **Artículo:** Yosinski et al. (2014), *How transferable are features in deep neural networks?*: mide qué capas se transfieren mejor.
- 📘 **Lectura:** Geirhos et al. (2019), *ImageNet-trained CNNs are biased towards texture*: explica el "panal" de la dalia.
- 🧰 **Herramienta:** `timm` (PyTorch Image Models): cientos de modelos preentrenados listos para *transfer learning* (para Colab).

---
⬅️ **Anterior:** [10.4 Arquitecturas icónicas: LeNet, AlexNet, VGG, ResNet](10.4-arquitecturas-iconicas.ipynb) · ➡️ **Siguiente:** [10.6 Más allá de clasificar: detección, segmentación y Vision Transformers](10.6-deteccion-segmentacion-y-vit.ipynb)